GUÍA PASO A PASO – API REST CON DATASET DE BIBLIOTECA
1️⃣ Preparar el entorno

Abrir el proyecto en Visual Studio Code

Crear carpeta del proyecto:

biblioteca-api

2️⃣ Crear entorno virtual

Abrir terminal y ejecutar:

    python -m venv venv

Activar entorno virtual.

Windows:

    venv\Scripts\activate

Mac/Linux:

    source venv/bin/activate

📒 ESTRUCTURA DEL NOTEBOOK

Crea un notebook llamado:

api_biblioteca_setup.ipynb

1️⃣ INSTALAR LIBRERÍAS

Primera celda:

    !pip install fastapi uvicorn pandas requests

Esto instala:

FastAPI

Uvicorn

pandas

requests

In [2]:
import pandas as pd

df = pd.read_csv("prestamos_biblioteca_publica.csv")
df.head()

,fecha_prestamo,categoria_libro,idioma,duracion_prestamo_dias,devuelto_a_tiempo,franja_horaria,mes,dia_semana
0,2025-01-05,ficcion,español,14,si,tarde,enero,domingo
1,2025-01-07,ensayo,español,21,si,mañana,enero,martes
2,2025-01-08,infantil,español,10,si,tarde,enero,miércoles
3,2025-01-10,comic,español,7,si,tarde,enero,viernes
4,2025-01-12,ficcion,inglés,15,no,mañana,enero,domingo


In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   fecha_prestamo          261 non-null    str  
 1   categoria_libro         261 non-null    str  
 2   idioma                  261 non-null    str  
 3   duracion_prestamo_dias  261 non-null    int64
 4   devuelto_a_tiempo       261 non-null    str  
 5   franja_horaria          261 non-null    str  
 6   mes                     261 non-null    str  
 7   dia_semana              261 non-null    str  
dtypes: int64(1), str(7)
memory usage: 16.4 KB


,duracion_prestamo_dias
count,261.000000
mean,14.896552
std,5.753998
min,7.000000
25%,10.000000
50%,14.000000
75%,19.000000
max,35.000000


5️⃣ Crear archivo de la API

Crear archivo:

main.py

In [4]:
api_code = """
from fastapi import FastAPI
import pandas as pd

app = FastAPI()

df = pd.read_csv("prestamos_biblioteca_publica.csv")

@app.get("/")
def home():
    return {"message": "API Biblioteca funcionando"}

@app.get("/prestamos")
def get_prestamos():
    return df.to_dict(orient="records")

@app.get("/prestamos/total")
def total_prestamos():
    return {"total_prestamos": len(df)}

@app.get("/prestamos/por-categoria")
def prestamos_categoria():
    result = df.groupby("categoria_libro").size().reset_index(name="total")
    return result.to_dict(orient="records")

@app.get("/prestamos/por-idioma")
def prestamos_idioma():
    result = df.groupby("idioma").size().reset_index(name="total")
    return result.to_dict(orient="records")

@app.get("/prestamos/por-mes")
def prestamos_mes():
    result = df.groupby("mes").size().reset_index(name="total")
    return result.to_dict(orient="records")

@app.get("/prestamos/duracion-media")
def duracion_media():
    return {"duracion_media": df["duracion_prestamo_dias"].mean()}
"""

with open("main.py", "w") as f:
    f.write(api_code)

In [6]:
import requests

url = "http://127.0.0.1:8000/prestamos"

data = requests.get(url).json()

df_api = pd.DataFrame(data)
df_api.head()

,fecha_prestamo,categoria_libro,idioma,duracion_prestamo_dias,devuelto_a_tiempo,franja_horaria,mes,dia_semana
0,2025-01-05,ficcion,español,14,si,tarde,enero,domingo
1,2025-01-07,ensayo,español,21,si,mañana,enero,martes
2,2025-01-08,infantil,español,10,si,tarde,enero,miércoles
3,2025-01-10,comic,español,7,si,tarde,enero,viernes
4,2025-01-12,ficcion,inglés,15,no,mañana,enero,domingo


In [7]:
requests.get("http://127.0.0.1:8000/prestamos/total").json()

{'total_prestamos': 261}

In [8]:
data = requests.get("http://127.0.0.1:8000/prestamos/por-categoria").json()
pd.DataFrame(data)

,categoria_libro,total
0,comic,35
1,divulgacion,14
2,ensayo,34
3,ficcion,103
4,infantil,55
5,novela_grafica,17
6,poesia,3


In [10]:
data = requests.get("http://127.0.0.1:8000/prestamos/por-idioma").json()
pd.DataFrame(data)

,idioma,total
0,catalán,29
1,español,209
2,inglés,23


In [11]:
df_api.groupby("categoria_libro")["duracion_prestamo_dias"].mean()

categoria_libro
comic              7.000000
divulgacion       25.000000
ensayo            22.558824
ficcion           14.873786
infantil          10.454545
novela_grafica    19.176471
poesia            31.000000
Name: duracion_prestamo_dias, dtype: float64